In [ ]:
import csv
import itertools
import logging
import os
import json
import random
import re
import shutil
import subprocess
import time
import threading
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED, Future
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Set, Dict, Iterable

import httpx
import undetected_chromedriver as uc
from dotenv import load_dotenv
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

load_dotenv()

BASE_DIR = Path("/home/kongla/Documents/GitHub/Real-estate-Scraping")
OUTPUT_PATH = BASE_DIR / "facebook" / "output.csv"
PROFILE_PATH = BASE_DIR / "chrome_profile"

MAX_STAGNANT = 10
SCROLL_SIZE = 3000
START_GROUP_IDX = 1

MODEL_NAME = "typhoon-v2.5-30b-a3b-instruct"
MAX_WORKERS = 18
LLM_TIMEOUT = 60.0
LLM_CONCURRENCY = 25
MAX_INFLIGHT_JOBS = 50
LLM_PAYLOAD_TRIM = 3500  # Trim payload to reduce TTFB latency
LLM_MAX_TOKENS = 12000

# Initialize httpx client for custom retry logic (no SDK storms)
_httpx_client = httpx.Client(
    timeout=LLM_TIMEOUT,
    limits=httpx.Limits(max_connections=MAX_WORKERS, max_keepalive_connections=5),
)

_clients = [
    OpenAI(api_key=os.getenv("TYPHOON_API_KEY"), base_url="https://api.opentyphoon.ai/v1", timeout=LLM_TIMEOUT),
    OpenAI(api_key=os.getenv("TYPHOON_API_KEY2"), base_url="https://api.opentyphoon.ai/v1", timeout=LLM_TIMEOUT),
    OpenAI(api_key=os.getenv("TYPHOON_API_KEY3"), base_url="https://api.opentyphoon.ai/v1", timeout=LLM_TIMEOUT),
]
_client_cycle = itertools.cycle(_clients)
_client_lock = threading.Lock()
_llm_semaphore = threading.Semaphore(LLM_CONCURRENCY)
_pending_lock = threading.Lock()
_pending_futures: Set[Future] = set()

def _get_client() -> OpenAI:
    with _client_lock:
        return next(_client_cycle)

def _register_future(fut: Future) -> None:
    with _pending_lock:
        _pending_futures.add(fut)

def _cleanup_done_futures() -> None:
    with _pending_lock:
        done = {f for f in _pending_futures if f.done()}
        _pending_futures.difference_update(done)

def _wait_for_backpressure() -> None:
    while True:
        _cleanup_done_futures()
        with _pending_lock:
            pending = len(_pending_futures)
        if pending < MAX_INFLIGHT_JOBS:
            return
        done, _ = wait(list(_pending_futures), return_when=FIRST_COMPLETED)
        with _pending_lock:
            _pending_futures.difference_update(done)

GROUP_URLS: List[str] = [
    "https://www.facebook.com/share/g/17yvNkWcJG/",
    "https://www.facebook.com/share/g/17cuAKX5Py/",
    "https://www.facebook.com/share/g/1Wm4GxnHFm/",
    "https://www.facebook.com/share/g/1Dp8rpYxmE/",
    "https://www.facebook.com/share/g/1F3cnPDmGU/",
    "https://www.facebook.com/share/g/1N2eXd6ge5/",
    "https://www.facebook.com/share/g/1AnUsnRNod/",
    "https://www.facebook.com/share/g/1JyWvsFtu6/",
    "https://www.facebook.com/share/g/1AzBzdPhEL/",
    "https://www.facebook.com/share/g/18NsbCsA52/",
    "https://www.facebook.com/share/g/1CKirSUbzs/",
    "https://www.facebook.com/share/g/173xAigqGb/",
    "https://www.facebook.com/share/g/17yZmR3H1U/",
    "https://www.facebook.com/share/g/1EV9v3JxPX/",
    "https://www.facebook.com/share/g/1BXjpfQN97/",
    "https://www.facebook.com/share/g/18Mh4scEdr/",
    "https://www.facebook.com/share/g/17rf5Ppt8a/",
    "https://www.facebook.com/share/g/1EGJYxCSeT/",
    "https://www.facebook.com/share/g/1C9S5cRLgw/",
    "https://www.facebook.com/share/g/1FgHtiV8oF/",
    "https://www.facebook.com/groups/485019775925280/",
    "https://www.facebook.com/share/g/1AgeQ1WpzA/",
    "https://www.facebook.com/share/g/1XYiLFYJgX/",
    "https://www.facebook.com/share/g/1CAAbCJJVT/",
    "https://www.facebook.com/share/g/1879ZUrbpH/",
    "https://www.facebook.com/share/g/1Bhed5kut5/",
    "https://www.facebook.com/share/g/1J75CA6Aox/",
    "https://www.facebook.com/share/g/1CVwGydoVb/",
    "https://www.facebook.com/share/g/1SKVJR95jx/",
    "https://www.facebook.com/share/g/1Di1RbGM5Z/",
    "https://www.facebook.com/share/g/1AgFYb1qH7/",
    "https://www.facebook.com/share/g/1WhmM13MbK/",
    "https://www.facebook.com/share/g/1Ch6CSbFLq/",
    "https://www.facebook.com/share/g/19vVjxviuP/",
    "https://www.facebook.com/share/g/189YXg4LZr/",
    "https://www.facebook.com/share/g/1HmmyqcGTq/",
    "https://www.facebook.com/share/g/18E2JPkYMH/",
    "https://www.facebook.com/share/g/1ArRmRHKEy/",
    "https://www.facebook.com/share/g/19c9HUsnBN/",
    "https://www.facebook.com/share/g/1GTM17dYew/",
    "https://www.facebook.com/share/g/189gkv2NVu/",
    "https://www.facebook.com/share/g/1BGgqYXFrJ/",
    "https://www.facebook.com/share/g/1E2rpaskGy/",
    "https://www.facebook.com/share/g/17iWwCY6Pc/",
    "https://www.facebook.com/share/g/1biieSTrft/",
    "https://www.facebook.com/share/g/1DmTmzs5Dn/",
    "https://www.facebook.com/share/g/1GDtgBoYjd/",
    "https://www.facebook.com/share/g/17o4QPYZfr/",
    "https://www.facebook.com/share/g/1AwtrUdMkw/",
    "https://www.facebook.com/share/g/1DFcTDtguE/",
    "https://www.facebook.com/share/g/14cTZ1dxvTo/",
    "https://www.facebook.com/share/g/1ENBMKVPrw/",
    "https://www.facebook.com/share/g/1Dksryieym/",
    "https://www.facebook.com/share/g/1DAyY5idaF/",
    "https://www.facebook.com/share/g/1CNm4wbqSe/",
    "https://www.facebook.com/share/g/17uw8vv7vb/",
    "https://www.facebook.com/share/g/1GPMEpyZEa/",
    "https://www.facebook.com/share/g/1DEHfe3cS3/",
    "https://www.facebook.com/share/g/1GZhnbeEVJ/",
    "https://www.facebook.com/share/g/1CUhdjzY2y/",
    "https://www.facebook.com/share/g/17G11FQ4MS/",
    "https://www.facebook.com/share/g/1Kmt1wQmrZ/",
    "https://www.facebook.com/share/g/1MhTuADusV/",
    "https://www.facebook.com/share/g/1CTofFNtcq/",
    "https://www.facebook.com/share/g/18A2nRTGhd/",
    "https://www.facebook.com/share/g/1AsSXshu4s/",
    "https://www.facebook.com/groups/175356699802854/",
    "https://www.facebook.com/share/g/18JNQPA5PU/",
    "https://www.facebook.com/share/g/18NPYCEess/",
    "https://www.facebook.com/share/g/1EKebYgt5Q/",
    "https://www.facebook.com/share/g/185ytLwwFg/",
    "https://www.facebook.com/share/g/1Dv3yJSrK4/",
    "https://www.facebook.com/share/g/17G6mCkmKJ/",
    "https://www.facebook.com/share/g/1FkNPpjiQT/",
    "https://www.facebook.com/share/g/1CK38hkJKi/",
    "https://www.facebook.com/share/g/1GTKcMsvPH/",
    "https://www.facebook.com/share/g/16kde1JhfD/",
    "https://www.facebook.com/share/g/1GdLnRnNa8/",
    "https://www.facebook.com/share/g/14VETLLJxFk/",
    "https://www.facebook.com/share/g/183AURyK31/",
    "https://www.facebook.com/share/g/1ATGortt8h/",
    "https://www.facebook.com/share/g/1briWbWRkE/",
    "https://www.facebook.com/share/g/18PynzrLZs/",
    "https://www.facebook.com/share/g/1Am4xiU2gK/",
    "https://www.facebook.com/share/g/17DJKPBtHd/",
    "https://www.facebook.com/groups/propertydit/",
    "https://www.facebook.com/share/g/185RyKmRPE/",
    "https://www.facebook.com/share/g/1CY3Mbb748/",
    "https://www.facebook.com/share/g/1B8oxbjTvm/",
    "https://www.facebook.com/share/g/1GSXJfPHRv/",
    "https://www.facebook.com/share/g/1M1HGCPGcu/",
    "https://www.facebook.com/share/g/1TNPq3ht9y/",
    "https://www.facebook.com/groups/2234744070053568/",
    "https://www.facebook.com/share/g/1CQGfxHye9/",
    "https://www.facebook.com/share/g/1DSPbvtBao/",
    "https://www.facebook.com/share/g/1GSvzLRBLr/",
    "https://www.facebook.com/share/g/1F8jgQaN2A/",
    "https://www.facebook.com/share/g/1HbNgxFjuL/",
    "https://www.facebook.com/share/g/1C88zkj62s/",
    "https://www.facebook.com/share/g/1Av5UuLqoH/",
    "https://www.facebook.com/share/g/1SmN6nuyk2/",
    "https://www.facebook.com/share/g/17H6RfoLRU/",
    "https://www.facebook.com/share/g/1cLKmAvPxn/",
    "https://www.facebook.com/share/g/174RKdW4tb/",
    "https://www.facebook.com/share/g/19WUdWcuDX/",
    "https://www.facebook.com/share/g/1Y2vcMpTCi/",
    "https://www.facebook.com/share/g/1CE4tovrbk/",
    "https://www.facebook.com/groups/1756597841090354/",
    "https://www.facebook.com/groups/theoneestate/",
    "https://www.facebook.com/groups/1512942549024788/",
    "https://www.facebook.com/share/g/1N8fiVtKft/",
    "https://www.facebook.com/share/g/17148ebYu2/",
    "https://www.facebook.com/share/g/1MeEAycCK9/",
    "https://www.facebook.com/share/g/1SZj2CdzZG/",
    "https://www.facebook.com/share/g/1J6mpUjhH1/",
    "https://www.facebook.com/share/g/1FBhAHeU2C/",
    "https://www.facebook.com/share/g/18PtvBe5Rk/",
    "https://www.facebook.com/share/g/1CV291GDMX/",
    "https://www.facebook.com/share/g/1ExkvVre1Z/",
    "https://www.facebook.com/groups/210390739740584/",
    "https://www.facebook.com/share/g/1E4S1igybj/",
    "https://www.facebook.com/share/g/1CKJ9gZFiE/",
    "https://www.facebook.com/share/g/1EMvFAXfAG/",
    "https://www.facebook.com/share/g/1BT3jt6Gzc/",
    "https://www.facebook.com/share/g/1AbuUgw43X/",
    "https://www.facebook.com/share/g/1GZJbNyZ7W/",
    "https://www.facebook.com/share/g/18GwgqjiBc/",
    "https://www.facebook.com/share/g/1HqpNUtH8B/",
    "https://www.facebook.com/share/g/1GXwMupTDL/",
    "https://www.facebook.com/share/g/1JTxjApoKP/",
    "https://www.facebook.com/share/g/19VteKNLoD/",
    "https://www.facebook.com/share/g/1C6x2GoH7u/",
    "https://www.facebook.com/share/g/1DaoFJMJkP/",
    "https://www.facebook.com/share/g/1DGSJAvihX/",
    "https://www.facebook.com/share/g/1AjZE6ZfSd/",
    "https://www.facebook.com/share/g/1DQok3tTUH/",
    "https://www.facebook.com/share/g/1C6oYUM8aD/",
    "https://www.facebook.com/share/g/1GEPncCTh9/",
    "https://www.facebook.com/share/g/1DjbxnhQBA/",
    "https://www.facebook.com/share/g/18Jr24GQVM/",
    "https://www.facebook.com/share/g/186TqCt9nn/",
    "https://www.facebook.com/share/g/1LKtYvTQyb/",
    "https://www.facebook.com/share/g/1HcXDi1ghz/",
    "https://www.facebook.com/share/g/1C1PfcEEU9/",
    "https://www.facebook.com/share/g/18JoUsmbRP/",
    "https://www.facebook.com/share/g/16wHaXjF3W/",
    "https://www.facebook.com/share/g/17AZ9awNPP/",
    "https://www.facebook.com/groups/770357389784713/",
    "https://www.facebook.com/share/g/1CB7iUsP7j/",
    "https://www.facebook.com/groups/condomarket/",
    "https://www.facebook.com/share/g/18EVu9X5hn/",
    "https://www.facebook.com/share/g/1KJZowXjYW/",
    "https://www.facebook.com/share/g/1Akg539Dgx/",
    "https://www.facebook.com/share/g/1L5WbUZqNR/",
    "https://www.facebook.com/groups/914277069917114/",
    "https://www.facebook.com/share/g/18MYU47ZQy/",
    "https://www.facebook.com/groups/1516977068738748/",
    "https://www.facebook.com/share/g/18akkmMoGG/",
]

OUTPUT_HEADERS = [
    "วันที่โพส", "website", "ประเภท", "สถานะ", "ชื่อโครงการ",
    "ขนาด", "ราคา", "เขต", "Link", "เบอร์โทรศัพท์", "Line", "คำอธิบาย"
]

SYSTEM_PROMPT = """คุณคือ AI Data Engine ระดับ Senior ที่เชี่ยวชาญด้าน Real Estate Analytics 
ทำหน้าที่สกัดข้อมูล (Entity Extraction) และจำแนกประเภท (Classification) จากข้อความโพสต์อสังหาริมทรัพย์
กฎเหล็ก: ตอบกลับเป็น JSON Structure เท่านั้น ห้ามมีข้อความเกริ่นนำหรือสรุปท้ายโดยเด็ดขาด 
**สำคัญมาก:** การขึ้นบรรทัดใหม่ในช่อง description ต้องใช้ escape character `\\n` เท่านั้น ห้ามกด Enter หรือขึ้นบรรทัดใหม่จริงๆ (Literal newlines) เพื่อป้องกัน JSON พัง

{
  "is_real_estate": true/false,
  "is_owner": true/false,
  "owner_confidence": 0.0,
  "classification_type": "OWNER/AGENT/UNKNOWN",
  "evidence_phrases": [],
  "risk_flags": [],
  "post_date_text": "สกัดข้อความวันที่/เวลาจากต้นฉบับ",
  "extracted":{
    "property_type": "บ้านเดี่ยว/คอนโด/ทาวน์โฮม/โฮมออฟฟิศ/ที่ดิน ฯลฯ",
    "rental_sale_status": "ขาย/เช่า/ขายและเช่า",
    "project_name": "ชื่อโครงการ (ถ้าไม่ระบุให้เป็น null)",
    "district": "เขต/พื้นที่/ทำเล/ถนนหลัก",
    "size_text": "ขนาดพื้นที่/พื้นที่ใช้สอย (รวมทั้ง ตร.ว. และ ตร.ม.)",
    "price_text": "ข้อความราคาเต็มพร้อมเงื่อนไข (เช่น 13.8 ล้าน ค่าโอน 50/50)",
    "price_value_thb": null,
    "phone": "เบอร์โทรศัพท์ (สกัดเฉพาะตัวเลขและขีด)",
    "line": "ID Line หรือ Link Line",
    "description": "ดึงรายละเอียดทั้งหมด ห้ามตัดทอน (ใช้ \\n แทนการขึ้นบรรทัดใหม่)"
  }
}

=== OWNER vs AGENT CLASSIFICATION LOGIC (EXPERT HEURISTICS) ===

ใช้กลไก Short-circuit Evaluation ตามลำดับความสำคัญดังนี้:

GATE 1: AGENT HARD-FILTER (ถือเป็น AGENT 100% ทันทีหากพบสิ่งเหล่านี้)
- [Line Official Account]: การระบุ LINE ID ที่มี "@" นำหน้า หรือมีข้อความกำกับเช่น "(มี @ ด้วย)", "(ใส่ @ เพิ่มเพื่อน)" ถือเป็น Agent แน่นอน (เช่น @likehome)
- [Bilingual Professional Template]: โพสต์ที่มีโครงสร้างจัดเต็ม 2 ภาษาสลับกันเป๊ะๆ (ไทย-อังกฤษ) พร้อมการแบ่งหมวดหมู่แบบมืออาชีพ (เช่น Property Details, Location Highlights, Contact)
- [Industry Keywords]: "ติดทรัพย์", "รับ Co-agent", "Co-broker", "ยินดีร่วมงานกับเอเจ้นท์", "รหัสทรัพย์", "Code:", "Stock"
- [Contact Patterns]: ชื่อผู้ติดต่อระบุว่าเป็น "Admin", "แอดมิน", "ทีมงาน", "ฝ่ายขาย", "Sale", หรือมี Link ให้กดติดตามเพจ/กลุ่ม
- [Corporate Tone & Closing]: "ปิดเกม", "Units สุดท้าย", "โค้งสุดท้าย", "จองด่วน", "Hot Item", "บริการด้านอสังหา", "ดันสินเชื่อทุกเคส"

GATE 2: OWNER VERIFIED (พิจารณาว่าเป็น OWNER ด้วย Confidence สูง 0.8-1.0)
- [Direct Explicit Claim]: "Owner Post", "เจ้าของขายเอง", "เจ้าของปล่อยเช่าเอง", "ยินดีรับ Agent" (กรณีบอกว่าตัวเองเป็นเจ้าของแล้วให้เอเจ้นท์ช่วยขาย)
- [Financial Loss/Transparency]: "ขายขาดทุน", "ราคาต่ำกว่าทุน", "ลดราคาจาก...", "จากราคา...เหลือ...", "ซื้อมาไม่ได้อยู่"
- [Personal Storytelling]: "ย้ายงาน", "ไปต่างประเทศ", "บ้านเรา", "แถมเครื่องใช้ไฟฟ้าให้หมด", "ตัดใจขาย" (มีบริบทร่วมกับชีวิตส่วนตัว)
- [Personal Tone]: ใช้สรรพนามบุคคล "พี่", "ผม", "ฉัน" อย่างเป็นธรรมชาติ ไม่เป็นแพทเทิร์น

GATE 3: HEURISTIC SCORING & AMBIGUITY HANDLING
- หากโพสต์มีลักษณะ "ข้อมูลพื้นฐานกระชับ สั้นมาก" (เช่น "ขายด่วน ขาดทุน 32ตรว. จาก 2.49ลบ. เหลือ 1.89ลบ.") ให้วิเคราะห์จากเนื้อหา หากมีความเป็น Financial Loss ชัดเจน ให้ถือเป็น OWNER (Confidence 0.7-0.9)
- แต่ถ้าโพสต์สั้นมากแบบ Generic (เช่น "ขายบ้าน 69 ล้าน บางนา กม.7 โทร 081-xxx") ไม่มี Personal Story เลย ให้ Default เป็น AGENT ด้วย Confidence ต่ำ (0.3 - 0.4) 
- ***กฎการทับซ้อน***: หากพบว่าเป็น "Owner Post" แต่วิเคราะห์เจอ LINE @ (Line OA) ให้ Short-circuit กลับไปเป็น AGENT ทันที (False Claim)

กฎการจัดการ Data Integrity:
1. price_value_thb: แปลงราคาเป็นตัวเลข Integer เพียวๆ (เช่น "1.89 ล้านบาท" -> 1890000, "110,000" -> 110000) หากประเมินไม่ได้ให้เป็น null
2. district: พยายามสกัดทำเลเด่นที่สุด เช่น "เสรีไทย", "รัชดา-ลาดพร้าว", "บางนาตราด กม.7"
"""

MONTH_MAP = {
    "มกราคม": 1, "ม.ค.": 1, "กุมภาพันธ์": 2, "ก.พ.": 2, "มีนาคม": 3, "มี.ค.": 3,
    "เมษายน": 4, "เม.ย.": 4, "พฤษภาคม": 5, "พ.ค.": 5, "มิถุนายน": 6, "มิ.ย.": 6,
    "กรกฎาคม": 7, "ก.ค.": 7, "สิงหาคม": 8, "ส.ค.": 8, "กันยายน": 9, "ก.ย.": 9,
    "ตุลาคม": 10, "ต.ค.": 10, "พฤศจิกายน": 11, "พ.ย.": 11, "ธันวาคม": 12, "ธ.ค.": 12,
}

_INVISIBLE_CHARS_RE = re.compile("[\u200b\u200c\u200d\ufeff\u00a0\u2060\u180e\u2028\u2029\u00ad]")
_TRUNCATION_SUFFIXES = ("... ดูเพิ่มเติม", "...ดูเพิ่มเติม", "... See more", "...See more")
csv_lock = threading.Lock()

# Agent phone numbers to filter (raw list provided by user)
AGENT_PHONE_LIST = [
    "061-3699614",
    "083-1524366",
    "081-6726652",
    "063-7803645",
    "065-5653642",
    "096-2636521",
    "092-3391919",
    "080-2475055",
    "062-9245628",
    "090-6919666",
    "065-8399388",
    "065-8516959",
    "095-2298853",
    "065-9722284",
    "089-4353098",
    "099-7879154",
    "098-8529884",
    "061-7584158",
    "094-1592265",
    "089-9525946",
    "080-1593057",
    "061-0028665",
    "089-7004546",
    "062-5521604",
    "094-7128014",
    "081-4344344",
    "093-0391151",
    "082-4122966",
    "086-4698186",
    "099-2681942",
    "062-2631715",
    "095-4461871",
    "098-9899053",
    "091-8515436",
    "097-9304489",
    "089-4353330",
    "098-6544946",
    "093-7495498",
    "096-5494597",
    "098-2852515",
    "099-4563645",
    "089-6331960",
    "095-6630685",
    "084-3497922",
    "096-3542925",
    "093-6495429",
    "065-9549746",
    "081-4024369",
    "086-9150863",
    "086-3115848",
    "093-1348881",
    "081-9638788",
    "099-3702992",
    "084-0022256",
    "095-1429241",
    "088-9794265",
    "092-6245663",
    "085-6257022",
    "061-8015669",
    "093-3179779",
    "062-9592146",
]

def _normalize_phone(s: str) -> str:
    if not s:
        return ""
    digits = re.sub(r"\\D", "", s)
    if digits.startswith("0"):
        digits = digits[1:]
    return digits

AGENT_PHONE_SET = {
    _normalize_phone(p) for p in AGENT_PHONE_LIST if p
}

def _phone_string_contains_banned(phone_str: str) -> bool:
    if not phone_str:
        return False
    for part in re.findall(r"\\d+", phone_str):
        if _normalize_phone(part) in AGENT_PHONE_SET:
            return True
    return False

def _is_truncated(content: str) -> bool:
    stripped = content.strip()
    return any(stripped.endswith(s) for s in _TRUNCATION_SUFFIXES)

def is_post_older_than_24h(date_text: str) -> bool:
    if not date_text or date_text == "N/A":
        return False
    val = _INVISIBLE_CHARS_RE.sub("", date_text).strip().lower()

    # ภายใน 24 ชม. ถือว่ายังอยู่ในกรอบเวลา
    if any(k in val for k in ("วันนี้", "นาที", "ชั่วโมง", "ชม.")):
        return False

    # ถ้าเจอเลขวัน >= 1 วัน (เช่น 2 วัน, 3 วัน) หรือเป็นวันที่แบบเดือน/ปี ถือว่าเกิน 24 ชม.
    m_days = re.search(r"(\d+)\s*วัน", val)
    if m_days:
        if int(m_days.group(1)) >= 1:
            return True

    m_date = re.search(r"(\d{1,2})[\s.\/-]+\w+", val)
    if m_date or "เมื่อวาน" in val:
        return True

    return False

def get_chrome_version(chrome_exec: str) -> int:
    try:
        res = subprocess.run([chrome_exec, "--version"], capture_output=True, text=True, check=False)
        return int(re.search(r"(\d+)\.", res.stdout).group(1)) if res.stdout else 0
    except Exception:
        return 0

def create_driver() -> uc.Chrome:
    PROFILE_PATH.mkdir(parents=True, exist_ok=True)
    chrome_exec = shutil.which("google-chrome") or shutil.which("chromium-browser")
    
    opts = uc.ChromeOptions()
    opts.add_argument(f"--user-data-dir={PROFILE_PATH}")
    
    # หั่น Payload ที่ไม่จำเป็นทิ้งทั้งหมด (Video, Audio, Image)
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.media_stream": 2,
        "profile.managed_default_content_settings.plugins": 2,
    }
    opts.add_experimental_option("prefs", prefs)
    
    opts.add_argument("--disable-notifications")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--headless=new")  # Use new headless mode for CPU efficiency
    opts.add_argument("--disable-features=IsolateOrigins,site-per-process")  # ลด Memory Footprint
    opts.add_argument("--blink-settings=imagesEnabled=false")
    opts.page_load_strategy = "eager"
    
    return uc.Chrome(options=opts, version_main=get_chrome_version(chrome_exec), browser_executable_path=chrome_exec)

def atomic_fb_extract(driver: uc.Chrome) -> tuple[List[Dict[str, str]], bool]:
    """
    Atomic JavaScript Pipeline: Scroll + Expand + Extract in single execute_script call
    Reduces WebDriver JSON-RPC overhead dramatically (~40-50% faster than sequential ops)
    Returns: (extracted_posts, found_old_post_flag)
    """
    result = driver.execute_script(r"""
        const INVIS_CODES = new Set([0x200B,0x200C,0x200D,0xFEFF,0x00A0,0x2060,0x180E,0x2028,0x2029,0x00AD]);
        function cleanText(s) {
            let out = '';
            for (let i = 0; i < s.length; i++) {
                if (!INVIS_CODES.has(s.charCodeAt(i))) out += s[i];
            }
            return out.trim().toLowerCase();
        }
        
        // STEP 1: Scroll to last article
        const articles = document.querySelectorAll("div[role='article']");
        if (articles.length > 0) {
            const lastArticle = articles[articles.length - 1];
            lastArticle.scrollIntoView({behavior: 'instant', block: 'end'});
            window.scrollBy(0, 800);
        }
        
        // STEP 2: Expand all "See more" buttons
        const TARGET = new Set(['ดูเพิ่มเติม', 'see more']);
        const candidates = Array.from(document.querySelectorAll('div[role="button"], span[role="button"]'));
        for (const el of candidates) {
            const text = cleanText(el.innerText || el.textContent || '');
            if (!TARGET.has(text)) continue;
            const rect = el.getBoundingClientRect();
            if (rect.width === 0 || rect.height === 0) continue;
            const cx = rect.left + rect.width / 2;
            const cy = rect.top + rect.height / 2;
            const evtOpts = {bubbles: true, cancelable: true, view: window, clientX: cx, clientY: cy, screenX: window.screenX + cx, screenY: window.screenY + cy};
            for (const evtType of ['pointerover','mouseover','pointermove','mousemove','pointerdown','mousedown','pointerup','mouseup','click']) {
                try { el.dispatchEvent(new MouseEvent(evtType, evtOpts)); } catch(_) {}
            }
        }
        
        // STEP 3: Extract all articles + early filter for old posts (24h boundary check)
        const results = [];
        let hasOldPost = false;
        document.querySelectorAll("div[role='article']").forEach(a => {
            const linkNodes = Array.from(a.querySelectorAll("a[href]")).filter(l => l.href.includes('/posts/') || l.href.includes('/permalink/'));
            if (linkNodes.length === 0) return;
            const url = linkNodes[0].href.split('?')[0];
            const msgNode = a.querySelector("div[data-ad-comet-preview='message']") || a.querySelector("div[data-ad-preview='message']");
            if (!msgNode) return;
            const content = msgNode.innerText.trim();
            let date = "N/A";
            for (let l of linkNodes) {
                const aria = (l.getAttribute("aria-label") || "").trim();
                const text = (l.textContent || "").trim();
                if (aria && aria.length > 0 && aria.length < 30) { date = aria; break; }
                else if (text && text.length > 0 && text.length < 30) { date = text; break; }
            }
            
            // Early boundary filtering: skip posts older than 24h at browser level
            const dateStr = cleanText(date);
            if (dateStr && !dateStr.includes('วันนี้') && !dateStr.includes('นาที') && !dateStr.includes('ชั่วโมง')) {
                const hasDayCount = /\d+\s*วัน/.test(dateStr);
                const hasDateFormat = /\d{1,2}[\s.\/-]+\w+/.test(dateStr);
                const isYesterday = dateStr.includes('เมื่อวาน');
                if (hasDayCount || hasDateFormat || isYesterday) {
                    hasOldPost = true;
                }
            }
            
            results.push({"Post_URL": url, "Full_Content": content, "Date": date});
        });
        return {results, hasOldPost};
    """)
    return result['results'], result['hasOldPost']

def humanized_scroll(driver: uc.Chrome) -> None:
    # Now handled by atomic_fb_extract - this is kept for fallback
    time.sleep(random.uniform(0.2, 0.3))

def apply_new_post_filter(driver: uc.Chrome):
    try:
        driver.execute_script("""
            const filterBtn = Array.from(document.querySelectorAll('div[role="button"]'))
                .find(e => e.innerText && (e.innerText.includes('\u0e40\u0e23\u0e35\u0e22\u0e07\u0e25\u0e33\u0e14\u0e31\u0e1a\u0e1f\u0e35\u0e14\u0e43\u0e19\u0e01\u0e25\u0e38\u0e48\u0e21\u0e15\u0e32\u0e21') || e.innerText.includes('\u0e08\u0e31\u0e14\u0e40\u0e23\u0e35\u0e22\u0e07\u0e15\u0e32\u0e21')));
            if (filterBtn) {
                filterBtn.click();
                setTimeout(() => {
                    const options = Array.from(document.querySelectorAll('div[role="menuitemradio"]'));
                    const target = options.find(e => e.innerText && (e.innerText.includes('\u0e42\u0e1e\u0e2a\u0e15\u0e4c\u0e43\u0e2b\u0e21\u0e48') || e.innerText.includes('\u0e23\u0e32\u0e22\u0e01\u0e32\u0e23\u0e2a\u0e34\u0e19\u0e04\u0e49\u0e32\u0e43\u0e2b\u0e21\u0e48')));
                    if (target) target.click();
                }, 1500);
            }
        """)
        time.sleep(3.5)
    except Exception as e:
        logger.error(f"Filter error: {e}")

def batch_extract_dom(driver: uc.Chrome) -> List[Dict[str, str]]:
    # Deprecated: use atomic_fb_extract instead
    return driver.execute_script("""
        const results = [];
        document.querySelectorAll("div[role='article']").forEach(a => {
            const linkNodes = Array.from(a.querySelectorAll("a[href]")).filter(l => l.href.includes('/posts/') || l.href.includes('/permalink/'));
            if (linkNodes.length === 0) return;
            const url = linkNodes[0].href.split('?')[0];
            const msgNode = a.querySelector("div[data-ad-comet-preview='message']") || a.querySelector("div[data-ad-preview='message']");
            if (!msgNode) return;
            const content = msgNode.innerText.trim();
            let date = "N/A";
            for (let l of linkNodes) {
                const aria = (l.getAttribute("aria-label") || "").trim();
                const text = (l.textContent || "").trim();
                if (aria && aria.length > 0 && aria.length < 30) { date = aria; break; }
                else if (text && text.length > 0 && text.length < 30) { date = text; break; }
            }
            results.push({"Post_URL": url, "Full_Content": content, "Date": date});
        });
        return results;
    """)

def call_llm_service(payload: str, raw_url: str = "") -> dict | None:
    """
    Strict HTTPX Client with custom retry logic (no SDK storms)
    Trimmed payload[:LLM_PAYLOAD_TRIM] to reduce TTFB latency
    """
    trimmed_payload = payload[:LLM_PAYLOAD_TRIM]
    
    for attempt in range(3):
        acquired = _llm_semaphore.acquire(timeout=LLM_TIMEOUT)
        if not acquired:
            logger.info(f"LLM semaphore timeout | URL: {raw_url} | attempt={attempt + 1}")
            time.sleep(1.0 * (attempt + 1))
            continue
        
        client_idx = None
        t0 = time.perf_counter()
        try:
            c = _get_client()
            client_idx = (_clients.index(c) + 1) if c in _clients else 0
            logger.info(f"LLM start | client={client_idx} | attempt={attempt + 1} | url={raw_url}")
            
            response = c.chat.completions.create(
                model=MODEL_NAME,
                temperature=0.0,
                max_tokens=LLM_MAX_TOKENS,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": trimmed_payload},
                ],
                response_format={"type": "json_object"},
            )
            content = response.choices[0].message.content
            elapsed = time.perf_counter() - t0
            logger.info(f"LLM ok | client={client_idx} | elapsed={elapsed:.2f}s | url={raw_url}")
            return json.loads(content)
        except Exception as exc:
            elapsed = time.perf_counter() - t0
            logger.error(f"LLM error | client={client_idx} | elapsed={elapsed:.2f}s | url={raw_url} | err={type(exc).__name__}: {exc}")
            time.sleep(2 ** attempt)
        finally:
            _llm_semaphore.release()
    return None

def parse_date(date_text: str) -> str:
    now = datetime.now()
    if not date_text or date_text == "N/A":
        return "-"
    val = _INVISIBLE_CHARS_RE.sub("", date_text).strip().lower()

    if any(k in val for k in ("วันนี้", "นาที", "ชั่วโมง", "ชม.")):
        return now.strftime("%d/%m/%Y")
    if "เมื่อวาน" in val:
        return (now - timedelta(days=1)).strftime("%d/%m/%Y")

    m_days = re.search(r"(\d+)\s*วัน", val)
    if m_days:
        return (now - timedelta(days=int(m_days.group(1)))).strftime("%d/%m/%Y")

    m_date = re.search(r"(\d{1,2})[\s.\/-]+\w+(?:[\s.\/-]+(\d{2,4}))?", val)
    if m_date:
        d, m_raw = int(m_date.group(1)), m_date.group(2) if m_date.lastindex >= 2 else None
        m = MONTH_MAP.get(m_raw) if m_raw else None
        if m:
            y = now.year
            if m_date.lastindex >= 3 and m_date.group(3):
                y_raw = int(m_date.group(3))
                y = y_raw - 543 if y_raw > 2400 else (y_raw if y_raw > 100 else 2000 + y_raw)
            return f"{d:02d}/{m:02d}/{y}"
    return "-"

def transform_record(raw_row: dict, ai_data: dict) -> dict:
    ext = ai_data.get("extracted", {})
    llm_date = _INVISIBLE_CHARS_RE.sub("", ai_data.get("post_date_text", "")).strip()
    dom_date = raw_row.get("Date", "").strip()
    actual_date = llm_date if llm_date and llm_date != "N/A" else dom_date
    return {
        "วันที่โพส": parse_date(actual_date),
        "website": "facebook",
        "ประเภท": ext.get("property_type", "-"),
        "สถานะ": ext.get("rental_sale_status", "-"),
        "ชื่อโครงการ": ext.get("project_name", "-"),
        "ขนาด": ext.get("size_text", "-"),
        "ราคา": ext.get("price_text", "-"),
        "เขต": ext.get("district", "-"),
        "Link": raw_row.get("Post_URL", "-"),
        "เบอร์โทรศัพท์": ext.get("phone", "-"),
        "Line": ext.get("line", "-"),
        "คำอธิบาย": ext.get("description", "-"),
    }

def worker_process_and_save(raw_item: dict) -> None:
    payload = f"Post Date: {raw_item.get('Date', 'N/A')}\n\nContent:\n{raw_item.get('Full_Content', '')}"
    ai_response = call_llm_service(payload, raw_item.get("Post_URL", ""))
    if not ai_response or not ai_response.get("is_real_estate"):
        return
    if not ai_response.get("is_owner"):
        logger.info(f"Agent Filtered | URL: {raw_item.get('Post_URL')} | Pattern: {ai_response.get('risk_flags')}")
        return
    final_data = transform_record(raw_item, ai_response)
    # Final agent-phone rule: drop records whose phone matches known agent contacts
    phone_field = final_data.get("เบอร์โทรศัพท์", "")
    if _phone_string_contains_banned(phone_field):
        logger.info(f"Agent Phone Filtered | URL: {raw_item.get('Post_URL')} | Phone: {phone_field}")
        return
    with csv_lock:
        with open(OUTPUT_PATH, "a", encoding="utf-8-sig", newline="") as f:
            csv.DictWriter(f, fieldnames=OUTPUT_HEADERS).writerow(final_data)


def process_group(driver: uc.Chrome, url: str, seen_urls: Set[str], executor: ThreadPoolExecutor, group_idx: int, total_groups: int):
    try:
        logger.info(f"[Group {group_idx}/{total_groups}] Start processing: {url}")
        driver.get(url)
        time.sleep(5)
        apply_new_post_filter(driver)

        saved_count, stagnant_count = 0, 0
        found_old_post = False
        for _ in range(300):
            _wait_for_backpressure()
            
            # Use atomic extraction pipeline (single JS call = no RPC overhead)
            extracted, found_old = atomic_fb_extract(driver)
            found_old_post = found_old_post or found_old
            
            if not extracted:
                stagnant_count += 1
            else:
                unseen = [i for i in extracted if i["Post_URL"] not in seen_urls]
                valid_unseen = [i for i in unseen if not is_post_older_than_24h(i["Date"])]
                new_items = [i for i in valid_unseen if not _is_truncated(i["Full_Content"])]
                truncated_count = len(valid_unseen) - len(new_items)

                if truncated_count:
                    logger.info(f"[Group {group_idx}/{total_groups}] Skipped {truncated_count} truncated post(s), will retry next iteration")

                if new_items:
                    stagnant_count = 0
                    for item in new_items:
                        seen_urls.add(item["Post_URL"])
                        fut = executor.submit(worker_process_and_save, item)
                        _register_future(fut)
                        saved_count += 1
                    logger.info(f"[Group {group_idx}/{total_groups}] Collected {len(new_items)} new items (Total saved this group: {saved_count})")
                elif truncated_count:
                    stagnant_count = 0
                else:
                    stagnant_count += 1

            if found_old_post or stagnant_count >= MAX_STAGNANT:
                reason = "Found post older than 24h" if found_old_post else f"Stagnant: {stagnant_count}"
                logger.info(f"[Group {group_idx}/{total_groups}] Stop condition met. ({reason}, Saved: {saved_count})")
                break

            humanized_scroll(driver)
    except Exception as e:
        logger.error(f"Error processing {url}: {e}")

def main():
    if not OUTPUT_PATH.exists():
        OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(OUTPUT_PATH, "w", newline="", encoding="utf-8-sig") as f:
            csv.DictWriter(f, fieldnames=OUTPUT_HEADERS).writeheader()

    seen_urls: Set[str] = set()
    with open(OUTPUT_PATH, "r", encoding="utf-8-sig") as f:
        seen_urls.update(row["Link"] for row in csv.DictReader(f) if row.get("Link"))

    total_groups = len(GROUP_URLS)
    resume_slice = GROUP_URLS[START_GROUP_IDX - 1:]
    logger.info(f"Resuming from group {START_GROUP_IDX}/{total_groups}: {resume_slice[0]}")

    driver = create_driver()
    try:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for idx, group_url in enumerate(resume_slice, start=START_GROUP_IDX):
                process_group(driver, group_url, seen_urls, executor, idx, total_groups)
            _wait_for_backpressure()
    finally:
        driver.quit()

if __name__ == "__main__":
    main()

[22:32:24] [INFO] Resuming from group 1/158: https://www.facebook.com/share/g/17yvNkWcJG/
[22:32:31] [INFO] patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
[22:32:31] [INFO] [Group 1/158] Start processing: https://www.facebook.com/share/g/17yvNkWcJG/


KeyboardInterrupt: 